In [1]:
# ===== Setup and frozen settings =====
import json
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

from spark_core import *

dev = pd.read_csv("../data/processed/v2_math_dev.csv")

with open("../outputs/tables/v2_best_params.json") as f:
    saved = json.load(f)

best_params = saved["best_params"]
print("Main model:", saved["main_model"], " Target recall:", TARGET_RECALL)

Main model: Random Forest  Target recall: 0.9


In [2]:
# ===== Build one frozen bundle per model =====
bundles = {}
for name in TUNING_GRIDS:
    estimators = {stage: make_pipeline(name, **best_params[stage][name])
                  for stage in STAGE_COLS}
    bundles[name] = fit_bundle(estimators, dev)

y_dev = dev["risk"].reset_index(drop=True)
is_high_dev = (y_dev == "High").astype(int).values
rows = []

for name, bundle in bundles.items():
    for stage, cols in STAGE_COLS.items():
        runs = oof_high_proba(make_pipeline(name, **best_params[stage][name]),
                              dev[cols], y_dev, n_repeats=3)
        flagged = [r >= bundle["thresholds"][stage] for r in runs]
        rows.append({
            "model": name, "stage": stage,
            "threshold": round(bundle["thresholds"][stage], 4),
            "share_flagged": np.mean([f.mean() for f in flagged]),
            "recall": np.mean([is_high_dev[f].sum() / is_high_dev.sum() for f in flagged]),
            "precision": np.mean([is_high_dev[f].mean() for f in flagged]),
        })

dev_operating = pd.DataFrame(rows).round(3)
dev_operating.to_csv("../outputs/tables/v2_dev_operating_points.csv", index=False)
dev_operating

,model,stage,threshold,share_flagged,recall,precision
0,Decision Tree,stage1,0.484,0.445,0.901,0.666
1,Decision Tree,stage2,0.598,0.351,0.904,0.847
2,Random Forest,stage1,0.496,0.445,0.901,0.666
3,Random Forest,stage2,0.562,0.350,0.904,0.849
4,KNN,stage1,0.424,0.458,0.920,0.661
5,KNN,stage2,0.543,0.362,0.923,0.840
6,Logistic regression,stage1,0.399,0.440,0.901,0.674
7,Logistic regression,stage2,0.562,0.351,0.904,0.847


In [3]:
# ===== Save the frozen bundles =====
for name, bundle in bundles.items():
    save_bundle(bundle, f"../models/v2_bundle_{name.replace(' ', '_')}.joblib")
print("Saved", len(bundles), "bundles")

Saved 4 bundles


In [4]:
# ===== Evaluation helper and Mathematics hold out =====
def run_eval(data, label):
    y_true = data["risk"].values
    rows = [
        {"test set": label, "approach": "Rule, G2 carried forward", "stage": "stage2",
         **evaluate_predictions(y_true, data["G2"].apply(assign_risk).values)},
        {"test set": label, "approach": "Rule, G1 carried forward", "stage": "stage1",
         **evaluate_predictions(y_true, data["G1"].apply(assign_risk).values)},
    ]
    for name, bundle in bundles.items():
        for stage, cols in STAGE_COLS.items():
            pred = score_students(data[cols], bundle)["risk"].values
            rows.append({"test set": label, "approach": name, "stage": stage,
                         **evaluate_predictions(y_true, pred)})

    out = pd.DataFrame(rows)
    out["n"] = len(data)
    out["high_students"] = int((y_true == "High").sum())
    return out.round(3)


holdout = pd.read_csv("../data/processed/v2_math_holdout_do_not_open.csv")
results_holdout = run_eval(holdout, "Mathematics hold out")
results_holdout.to_csv("../outputs/tables/v2_holdout_results.csv", index=False)

for stage, cols in STAGE_COLS.items():
    pred = score_students(holdout[cols], bundles["Random Forest"])["risk"].values
    print("Random Forest,", stage)
    print(pd.DataFrame(
        confusion_matrix(holdout["risk"], pred, labels=RISK_ORDER),
        index=["true " + r for r in RISK_ORDER],
        columns=["pred " + r for r in RISK_ORDER]))
    print()

results_holdout

Random Forest, stage1
             pred High  pred Medium  pred Low
true High           24            2         0
true Medium         14           17         2
true Low             0            6        14

Random Forest, stage2
             pred High  pred Medium  pred Low
true High           24            2         0
true Medium          5           26         2
true Low             0            2        18



,test set,approach,stage,recall_high,precision_high,f1_macro,accuracy,flagged_high,high_predicted_low,n,high_students
0,Mathematics hold out,"Rule, G2 carried forward",stage2,0.923,0.800,0.869,0.861,30,0,79,26
1,Mathematics hold out,"Rule, G1 carried forward",stage1,0.808,0.750,0.754,0.747,28,0,79,26
2,Mathematics hold out,Decision Tree,stage1,0.923,0.632,0.705,0.696,38,0,79,26
3,Mathematics hold out,Decision Tree,stage2,0.923,0.800,0.869,0.861,30,0,79,26
4,Mathematics hold out,Random Forest,stage1,0.923,0.632,0.705,0.696,38,0,79,26
5,Mathematics hold out,Random Forest,stage2,0.923,0.828,0.866,0.861,29,0,79,26
6,Mathematics hold out,KNN,stage1,0.808,0.750,0.754,0.747,28,0,79,26
7,Mathematics hold out,KNN,stage2,0.885,0.852,0.853,0.848,27,0,79,26
8,Mathematics hold out,Logistic regression,stage1,0.808,0.750,0.754,0.747,28,0,79,26
9,Mathematics hold out,Logistic regression,stage2,0.885,0.852,0.866,0.861,27,0,79,26


In [5]:
# ===== Portuguese cross subject test =====
por_full, por_clean = load_portuguese()

results_por = pd.concat([
    run_eval(por_clean, "Portuguese, no overlap"),
    run_eval(por_full, "Portuguese, full file"),
])
results_por.to_csv("../outputs/tables/v2_portuguese_results.csv", index=False)

print(por_clean["risk"].value_counts().reindex(RISK_ORDER).to_dict())
results_por

{'High': 70, 'Medium': 141, 'Low': 64}


,test set,approach,stage,recall_high,precision_high,f1_macro,accuracy,flagged_high,high_predicted_low,n,high_students
0,"Portuguese, no overlap","Rule, G2 carried forward",stage2,0.900,0.670,0.797,0.793,94,0,275,70
1,"Portuguese, no overlap","Rule, G1 carried forward",stage1,0.900,0.600,0.690,0.695,105,0,275,70
2,"Portuguese, no overlap",Decision Tree,stage1,1.000,0.467,0.597,0.582,150,0,275,70
3,"Portuguese, no overlap",Decision Tree,stage2,0.900,0.670,0.797,0.793,94,0,275,70
4,"Portuguese, no overlap",Random Forest,stage1,1.000,0.467,0.597,0.582,150,0,275,70
5,"Portuguese, no overlap",Random Forest,stage2,0.829,0.674,0.788,0.785,86,0,275,70
6,"Portuguese, no overlap",KNN,stage1,0.900,0.600,0.690,0.695,105,0,275,70
7,"Portuguese, no overlap",KNN,stage2,0.900,0.708,0.808,0.807,89,0,275,70
8,"Portuguese, no overlap",Logistic regression,stage1,0.900,0.600,0.717,0.709,105,0,275,70
9,"Portuguese, no overlap",Logistic regression,stage2,0.900,0.708,0.810,0.807,89,0,275,70
